# 새 영상 2종 검증 — Baseline vs Tuned 비교

**목표:** 튜닝 파라미터(conf=0.25, iou=0.7, imgsz=800)가 새 영상에서도 효율적인지 검증

| 구분 | conf | iou | imgsz | 설명 |
|------|------|-----|-------|------|
| **Baseline** | 0.25 | 0.70 | 640 | YOLOv8L 기본값 (imgsz만 기본값) |
| **Tuned** | 0.25 | 0.70 | 800 | 튜닝 최적값 (F1=0.8014) |

> **비교 포인트:** conf·iou는 동일, **imgsz 640 vs 800 차이만** 순수하게 비교

**검증 대상 영상:**
- `SampleVideo_Scenes3/` — 자취방 영상 (223클립)
- `SampleVideo_Scenes2/` — 요리 영상 (179클립)

**실행 전 체크리스트:**
- [ ] 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
- [ ] Google Drive에 아래 폴더 업로드 확인
  - `내 드라이브/SampleVideo.Test/SampleVideo_Scenes3/` (223개 mp4)
  - `내 드라이브/SampleVideo.Test/SampleVideo_Scenes2/` (179개 mp4)

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
# 패키지 설치
!pip install -q ultralytics
print("ultralytics 설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# 경로 설정
BASE_DRIVE = "/content/drive/MyDrive/SampleVideo.Test"

VIDEO_SETS = {
    "자취방": os.path.join(BASE_DRIVE, "SampleVideo_Scenes3"),
    "요리":   os.path.join(BASE_DRIVE, "SampleVideo_Scenes2"),
}

EXPECTED_CLIPS = {"자취방": 223, "요리": 179}

for name, path in VIDEO_SETS.items():
    clips = len([f for f in os.listdir(path) if f.endswith('.mp4')])
    status = "OK" if clips == EXPECTED_CLIPS[name] else f"주의: {clips}개 (예상 {EXPECTED_CLIPS[name]}개)"
    print(f"[{name}] {clips}개 클립 — {status}")

print("경로 확인 완료!")

## 1. 실험 설정

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from tqdm.auto import tqdm

# ========================
# 실험 구성
# ========================
MODEL_PATH = "yolov8l.pt"
DATA_YAML  = "coco128.yaml"

# MVP 15 class ids
MVP_CLASS_IDS = [16, 26, 39, 41, 45, 53, 56, 57, 59, 60, 62, 63, 65, 67, 72]

CONFIGS = [
    {"label": "Baseline", "conf": 0.25, "iou": 0.70, "imgsz": 640},  # YOLOv8 default
    {"label": "Tuned",    "conf": 0.25, "iou": 0.70, "imgsz": 800},  # 튜닝 최적값
]

total_experiments = len(CONFIGS) * len(VIDEO_SETS)

# 모델 로드
print("Loading YOLOv8L model...")
model = YOLO(MODEL_PATH)
print("모델 로드 완료!")
print()
print("=" * 55)
print(f"  실험 구성: {len(CONFIGS)}개 Config × {len(VIDEO_SETS)}개 영상 = 총 {total_experiments}회")
print(f"  비교 포인트: imgsz 640(Baseline) vs 800(Tuned) — conf/iou 동일")
for cfg in CONFIGS:
    print(f"  [{cfg['label']:8s}] conf={cfg['conf']}, iou={cfg['iou']}, imgsz={cfg['imgsz']}")
print("=" * 55)

## 2. COCO128 Validation (Precision / Recall / F1)

In [ ]:
val_rows = []

for cfg in CONFIGS:
    print(f"\n[{cfg['label']}] conf={cfg['conf']}, iou={cfg['iou']}, imgsz={cfg['imgsz']}")
    t0 = time.time()
    metrics = model.val(
        data=DATA_YAML,
        imgsz=cfg["imgsz"],
        conf=cfg["conf"],
        iou=cfg["iou"],
        classes=MVP_CLASS_IDS,
        verbose=False
    )
    elapsed = time.time() - t0

    precision_arr = metrics.box.p
    recall_arr    = metrics.box.r
    precision = float(np.nanmean(precision_arr)) if hasattr(precision_arr, "__len__") else float(precision_arr)
    recall    = float(np.nanmean(recall_arr))    if hasattr(recall_arr,    "__len__") else float(recall_arr)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"  Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f} ({elapsed:.1f}s)")
    val_rows.append({
        "Config":    cfg["label"],
        "conf":      cfg["conf"],
        "iou":       cfg["iou"],
        "imgsz":     cfg["imgsz"],
        "Precision": round(precision, 4),
        "Recall":    round(recall, 4),
        "F1":        round(f1, 4),
        "val_time(s)": round(elapsed, 1),
    })

df_val = pd.DataFrame(val_rows)
print("\n=== COCO128 Validation 결과 ===")
print(df_val.to_string(index=False))

## 3. 새 영상 추론 (탐지 수 비교)

In [ ]:
infer_rows = []
exp_num = 0

for video_name, video_path in VIDEO_SETS.items():
    scene_files = sorted(Path(video_path).glob("*.mp4"))

    for cfg in CONFIGS:
        exp_num += 1
        print(f"\n{'='*55}")
        print(f"  [{exp_num}/{total_experiments}] [{cfg['label']}] {video_name} ({len(scene_files)}클립)")
        print(f"  conf={cfg['conf']}, iou={cfg['iou']}, imgsz={cfg['imgsz']}")
        print(f"{'='*55}")

        total_detected = 0
        t0 = time.time()

        pbar = tqdm(scene_files, desc=f"  {cfg['label']} / {video_name}", leave=True)
        for scene_file in pbar:
            results = model.predict(
                source=str(scene_file),
                conf=cfg["conf"],
                iou=cfg["iou"],
                imgsz=cfg["imgsz"],
                classes=MVP_CLASS_IDS,
                stream=True,
                verbose=False
            )
            for result in results:
                if result.boxes is not None:
                    total_detected += len(result.boxes)
            pbar.set_postfix({"detected": total_detected})

        elapsed = time.time() - t0
        det_per_sec = total_detected / elapsed if elapsed > 0 else 0

        print(f"  => 탐지: {total_detected}개 | 시간: {elapsed:.1f}s | {det_per_sec:.1f} det/s")

        infer_rows.append({
            "영상":        video_name,
            "Config":     cfg["label"],
            "conf":       cfg["conf"],
            "iou":        cfg["iou"],
            "imgsz":      cfg["imgsz"],
            "클립 수":     len(scene_files),
            "탐지 수":     total_detected,
            "소요 시간(s)": round(elapsed, 1),
            "탐지/sec":    round(det_per_sec, 1),
        })

df_infer = pd.DataFrame(infer_rows)
print("\n모든 추론 완료!")

## 4. 결과 확인

In [ ]:
from IPython.display import display, HTML

# ── COCO128 Validation 비교
baseline_f1 = df_val[df_val['Config']=='Baseline']['F1'].values[0]
tuned_f1    = df_val[df_val['Config']=='Tuned']['F1'].values[0]
f1_diff     = tuned_f1 - baseline_f1

html = "<h3>COCO128 Validation — Baseline vs Tuned</h3>"
html += "<table border='1' style='border-collapse:collapse; width:600px; font-size:14px;'>"
html += "<tr style='background:#ddd;'><th>Config</th><th>conf</th><th>iou</th><th>imgsz</th><th>Precision</th><th>Recall</th><th>F1</th></tr>"
for _, row in df_val.iterrows():
    bg = '#E0FFE0' if row['Config'] == 'Tuned' else '#FFFFFF'
    html += f"<tr style='background:{bg};'>"
    html += f"<td><b>{row['Config']}</b></td><td>{row['conf']}</td><td>{row['iou']}</td><td>{row['imgsz']}</td>"
    html += f"<td>{row['Precision']:.4f}</td><td>{row['Recall']:.4f}</td><td><b>{row['F1']:.4f}</b></td></tr>"
html += "</table>"
html += f"<p><b>F1 향상: {f1_diff:+.4f}</b> (Tuned - Baseline)</p>"

# ── 영상별 탐지 수 비교
html += "<h3>새 영상 탐지 수 비교</h3>"
html += "<table border='1' style='border-collapse:collapse; width:700px; font-size:14px;'>"
html += "<tr style='background:#ddd;'><th>영상</th><th>Config</th><th>클립 수</th><th>탐지 수</th><th>소요 시간(s)</th><th>탐지/sec</th><th>클립당 탐지</th></tr>"
for _, row in df_infer.iterrows():
    bg = '#E0FFE0' if row['Config'] == 'Tuned' else '#FFFFFF'
    per_clip = round(row['탐지 수'] / row['클립 수'], 1) if row['클립 수'] > 0 else 0
    html += f"<tr style='background:{bg};'>"
    html += f"<td>{row['영상']}</td><td><b>{row['Config']}</b></td><td>{row['클립 수']}</td>"
    html += f"<td><b>{row['탐지 수']:,}</b></td><td>{row['소요 시간(s)']}</td><td>{row['탐지/sec']}</td><td>{per_clip}</td></tr>"
html += "</table>"

# ── 영상별 탐지 수 증감
html += "<h3>영상별 Tuned 효과 (탐지 수 변화)</h3>"
for vname in VIDEO_SETS.keys():
    b = df_infer[(df_infer['영상']==vname) & (df_infer['Config']=='Baseline')]['탐지 수'].values[0]
    t = df_infer[(df_infer['영상']==vname) & (df_infer['Config']=='Tuned')]['탐지 수'].values[0]
    diff = t - b
    pct  = diff / b * 100 if b > 0 else 0
    color = '#006600' if diff >= 0 else '#cc0000'
    html += f"<p><b>{vname}</b>: Baseline={b:,} → Tuned={t:,} "
    html += f"(<span style='color:{color};'>{diff:+,} / {pct:+.1f}%</span>)</p>"

display(HTML(html))

## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
import os

# ── Colab 한글 폰트 설치 및 적용
os.system("apt-get install -y fonts-nanum > /dev/null 2>&1")

# 폰트 캐시 갱신
fm._load_fontmanager(try_read_cache=False)

# NanumGothic 경로 탐색
nanum_fonts = [f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name]
if nanum_fonts:
    font_path = nanum_fonts[0]
    font_prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = font_prop.get_name()
    print(f"한글 폰트 적용: {font_prop.get_name()}")
else:
    # fallback: 경로 직접 지정
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
    fm.fontManager.addfont(font_path)
    font_prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = font_prop.get_name()
    print(f"한글 폰트 직접 등록: {font_path}")

matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Baseline vs Tuned — New Video Validation', fontsize=14, fontweight='bold')

colors = {'Baseline': 'steelblue', 'Tuned': 'tomato'}
video_labels = list(VIDEO_SETS.keys())

# ── 왼쪽: COCO128 F1 비교
ax = axes[0]
configs = df_val['Config'].tolist()
f1_vals = df_val['F1'].tolist()
bars = ax.bar(configs, f1_vals,
              color=[colors[c] for c in configs],
              alpha=0.85, width=0.5)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('COCO128 F1 (Ground Truth)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# ── 가운데: 영상별 탐지 수 비교
ax = axes[1]
x = range(len(video_labels))
width = 0.35
for i, cfg_label in enumerate(['Baseline', 'Tuned']):
    vals = [df_infer[(df_infer['영상']==v) & (df_infer['Config']==cfg_label)]['탐지 수'].values[0]
            for v in video_labels]
    bars = ax.bar([xi + i*width for xi in x], vals,
                  width=width, color=colors[cfg_label], alpha=0.85, label=cfg_label)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{val:,}', ha='center', fontsize=9)
ax.set_xticks([xi + width/2 for xi in x])
ax.set_xticklabels(video_labels, fontsize=11, fontproperties=font_prop)
ax.set_ylabel('Total Detections', fontsize=12)
ax.set_title('Detection Count by Video', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# ── 오른쪽: 클립당 평균 탐지 수
ax = axes[2]
for i, cfg_label in enumerate(['Baseline', 'Tuned']):
    vals = []
    for v in video_labels:
        row = df_infer[(df_infer['영상']==v) & (df_infer['Config']==cfg_label)].iloc[0]
        vals.append(round(row['탐지 수'] / row['클립 수'], 1))
    bars = ax.bar([xi + i*width for xi in x], vals,
                  width=width, color=colors[cfg_label], alpha=0.85, label=cfg_label)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', fontsize=9)
ax.set_xticks([xi + width/2 for xi in x])
ax.set_xticklabels(video_labels, fontsize=11, fontproperties=font_prop)
ax.set_ylabel('Avg Detections / Clip', fontsize=12)
ax.set_title('Avg Detections per Clip', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('validate_new_videos_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: validate_new_videos_result.png")

## 6. 결과 저장 & 다운로드

In [ ]:
import shutil
drive_save_dir = "/content/drive/MyDrive/SampleVideo.Test"

# CSV 저장
df_val.to_csv("validate_coco128_results.csv",   index=False, encoding='utf-8-sig')
df_infer.to_csv("validate_infer_results.csv",   index=False, encoding='utf-8-sig')

# Drive 백업
df_val.to_csv(f"{drive_save_dir}/validate_coco128_results.csv",  index=False, encoding='utf-8-sig')
df_infer.to_csv(f"{drive_save_dir}/validate_infer_results.csv",  index=False, encoding='utf-8-sig')
shutil.copy('validate_new_videos_result.png', f'{drive_save_dir}/validate_new_videos_result.png')
print("Drive 백업 완료")

# 다운로드
from google.colab import files
files.download('validate_coco128_results.csv')
files.download('validate_infer_results.csv')
files.download('validate_new_videos_result.png')
print("다운로드 시작!")

In [ ]:
# Report용 마크다운 출력
print("=" * 60)
print("  아래 내용을 Project_Report.md에 붙여넣으세요")
print("=" * 60)
print()
print("### COCO128 Validation")
print(df_val[['Config','conf','iou','imgsz','Precision','Recall','F1']].to_markdown(index=False))
print()
print("### 새 영상 추론 결과")
print(df_infer[['영상','Config','클립 수','탐지 수','소요 시간(s)','탐지/sec']].to_markdown(index=False))
print()

for vname in VIDEO_SETS.keys():
    b = df_infer[(df_infer['영상']==vname) & (df_infer['Config']=='Baseline')]['탐지 수'].values[0]
    t = df_infer[(df_infer['영상']==vname) & (df_infer['Config']=='Tuned')]['탐지 수'].values[0]
    diff = t - b
    pct  = diff / b * 100 if b > 0 else 0
    print(f"★ {vname}: Baseline={b:,} → Tuned={t:,} ({diff:+,} / {pct:+.1f}%)")